Берем все из предыдущего урока

In [4]:
import pandas as pd
# Читаем файл melb_data.csv
melb_data = pd.read_csv('data/melb_data.csv', sep=',')

melb_df = melb_data.copy()
melb_df['Date'] = pd.to_datetime(melb_df['Date'], dayfirst=True)
years_sold = melb_df['Date'].dt.year
melb_df['MonthSale'] = melb_df['Date'].dt.month
delta_days = melb_df['Date'] - pd.to_datetime('2016-01-01')
melb_df['AgeBuilding'] = melb_df['Date'].dt.year - melb_df['YearBuilt']
melb_df['AgeBuilding'].astype('int16')
melb_df = melb_df.drop('YearBuilt', axis=1)
melb_df['WeekdaySale'] = melb_df['Date'].dt.dayofweek
weekend_count = melb_df[(melb_df['WeekdaySale'] == 5) | (melb_df['WeekdaySale'] == 6)].shape[0]

# На вход функции поступает строка с адресом
def get_street_type(adress):
    stype_dict = {
        'Avenue': 'Ave',
        'Boulevard': 'Bvd',
        'Parade': 'Pde'
    }
# Создаем список географических отметок
    exclude_list = ['N', 'S', 'W', 'E']
# Метод split() разбивает строку по пробелу
# В результате получим список слов в строке и заносим его в переменную adress_list
    adress_list = adress.split(' ')
# Обрезаем список, оставляя в нем только последний элемент,
# потенциальный подтип улицы, и заносим в переменную street_type
    street_type = adress_list[-1]
# Проверяем, что полученный подтип является географической пометкой.
# Для этого проверяем его на наличие в списке exclude_list
    if street_type in exclude_list:
# Если street_type в exclude_list (является географической пометкой),
# переопределяем ее на второй элемент с конца списка adress_list
        street_type = adress_list[-2]
    street_type = stype_dict.get(street_type, street_type)
# Возвращаем street_type, в котором хранится подтип улицы
    return street_type


street_types = melb_df['Address'].apply(get_street_type)

popular_stypes = street_types.value_counts().nlargest(n=10).index

melb_df['StreetType'] = street_types.apply(lambda x: x if x in popular_stypes else 'other')

melb_df = melb_df.drop('Address', axis=1)

# Создаем функцию, которая принимает в качестве аргумента элемент столбца WeekdaySale,
# возвращает 1, если день выходной, и 0 — если не выходной
def get_weekend(weekday):
    if weekday == 5 or weekday == 6:
        return 1
    return 0
# Создаем новый столбец Weekend, применив функцию к столбцу WeekdaySale
melb_df['Weekend'] = melb_df['WeekdaySale'].apply(get_weekend)
# Вычисляем среднюю цену объекта недвижимости, проданного в выходные дни
wknd_mean = melb_df[melb_df['Weekend'] == 1]['Price'].mean()

# Выделяем 49 самых популярных селлеров
popular_sellers = melb_df['SellerG'].value_counts().nlargest(n=49).index
# Преобразуем столбец SellerG, меняя всех, кто не входит в число 49 популярных селлеров на other
melb_df['SellerG'] = melb_df['SellerG'].apply(lambda popular: popular if popular in popular_sellers else 'other')

Под числовыми признаками обычно подразумевают признаки, отражающие количественную меру и которые могут принимать значения из неограниченного диапазона.

Числовые признаки могут быть:

- Дискретными (например, количество комнат, пациентов, дней);
- Непрерывными (масса, цена, площадь).

Дискретные признаки чаще всего представлены целыми числами, а непрерывные — целыми числами и числами с плавающейй точкой.

Под категориальными признаками обычно подразумевают столбцы в таблице, которые обозначают принадлежность объекта к какому-то классу/категории.

Категориальные признаки могут быть:

- Номинальными (пол, национальность, район);
- Порядковыми (уровень образования, уровень комфорта, стадия заболевания).

Такие признаки имеют ограниченный набор значений. Они чаще всего представлены в виде текстового описания и кодируются в Pandas типом данных object.

ОДНАКО это не всегда так. Например, признак месяца продажи объекта недвижимости кодируется числом (от 1 до 12), но на самом деле является категориальным, поскольку диапазон его значений ограничен и каждому числу мы можем поставить в соответствие название месяца.

Решение, какой признак отнести к классу категорий, остается за исследователем. Некоторые специалисты даже относят количественные признаки в разряд категориальных, если диапазон возможных значений слишком мал.

Анализ и предобработка категориальных признаков отличается от предобработки числовых признаков.

В столбцах с типом данных object мы не можем рассчитать среднее, стандартное отклонение или другие статистические параметры, которые годятся только для чисел.

Способ заполнения пропущенных данных и поиск аномальных значений также различаются для разных типов признаков.

Категории в данных о недвижимости

Необходимо определить число уникальных категорий в каждом столбце таблицы melb_df. Для этого нужно создать вспомогательную таблицу unique_counts:

In [5]:
# Создаем пустой список
unique_list = []
# Проходим по именам столбцов в таблице
for col in melb_df.columns:
    # Создаем кортеж (имя столбца, число уникальных значений)
    item = (col, melb_df[col].nunique(),melb_df[col].dtypes) 
    # добавляем кортеж в список
    unique_list.append(item)
# Создаем вспомогательную таблицу и сортируем ее
unique_counts = pd.DataFrame(
    unique_list,
    columns=['Column_Name', 'Num_Unique', 'Type']
).sort_values(by='Num_Unique', ignore_index=True)
# Выводим таблицу на экран
display(unique_counts)


,Column_Name,Num_Unique,Type
0,Weekend,2,int64
1,Type,3,object
2,WeekdaySale,5,int32
3,Method,5,object
4,Regionname,8,object
5,Rooms,9,int64
6,Bathroom,9,float64
7,StreetType,11,object
8,Car,11,float64
9,Bedroom,12,float64


1. Создаем пустой список, в который будем добавлять кортежи: имя столбца, число уникальных значений в нем и тип столбца;
2. В цикле перебираем имена столбцов, которые получаем с помощью атрибута columns. В переменной col на каждой итерации находятся имена столбцов — обращаемся к ним в цикле и извлекаем число уникальных элементов с помощью nunique(), а также тип столбца с помощью атрибута dtypes(). Результат заносим в кортеж и добавляем его в список.
3. Из списка с кортежами (имя столбца, число уникальных значений, тип столбца) создаем датафрейм, даем названия его столбцам;
4. Сортируем таблицу по столбцу Num_unique в порядке возрастания количества уникальных элементов и выводим результат на экран.

ПРИМЕЧАНИЕ: сортировка sort_values() будет изучена позже.

Если присмотреться, можно увидеть резкий скачок количества уникальных значений, начиная с 14 строки таблицы, где число уникальных значений составляет 152.

Категориальными будем считать признаки, у которых число уникальных категорий меньше 150.

ОДНАКО признак Date (дата продажи), преобразованный ранее в формат datetime, является временным признаком, поэтому далее не будем воспринимать его как категориальный.

К тому же в потенциальный список попали количественные столбцы Rooms, Car, Bedroom, Bathroom. Не будем относить их к разряду категориальных признаков, но такое тоже вполне возможно.

Такая классификация признаков является исключительно субъективной и специфична для задачи.

Для хранения и оптимизации работы с категориальными признаками в Pandas предусмотрен специальный тип данных — category.

Этот тип данных является гибридным: внешне выглядит как строка, но внутренее представлен массивом целых чисел. Так как данные вместо изначальных строк хранятся в памяти как число, то объем памяти, занимаемой таблицей при использовании типа category, резко уменьшается, что повышает эффективность хранения и работы с таблицей.

Также этот тип расширяет возможности работы с категориальными признаками: можно легко преобразовывать категории, строить графики по таким данным (это сложно сделать для типа object). Также резко повышается производительность операций, совершаемых с такими столбцами.

Самый простой способ преобразования столбцов к типу данных category — это использование метода astype(), в параметры которого достаточно передать 'category'.

In [6]:
display(melb_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   index          13580 non-null  int64         
 1   Suburb         13580 non-null  object        
 2   Rooms          13580 non-null  int64         
 3   Type           13580 non-null  object        
 4   Price          13580 non-null  float64       
 5   Method         13580 non-null  object        
 6   SellerG        13580 non-null  object        
 7   Date           13580 non-null  datetime64[ns]
 8   Distance       13580 non-null  float64       
 9   Postcode       13580 non-null  int64         
 10  Bedroom        13580 non-null  float64       
 11  Bathroom       13580 non-null  float64       
 12  Car            13580 non-null  float64       
 13  Landsize       13580 non-null  float64       
 14  BuildingArea   13580 non-null  float64       
 15  CouncilArea    1221

None

In [7]:
# Делаем преобразование столбцов к типу category
cols_to_exclude = ['Date', 'Rooms', 'Bedroom', 'Bathroom', 'Car'] # список столбцов, которые не берем во внимание
max_unique_count = 150 # задаем максимальное количество уникальных категорий
for col in melb_df.columns: # цикл по именам столбцов
    if melb_df[col].nunique() < max_unique_count and col not in cols_to_exclude: # проверяем условие
        melb_df[col] = melb_df[col].astype('category') # преобразуем тип столбца
display(melb_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   index          13580 non-null  int64         
 1   Suburb         13580 non-null  object        
 2   Rooms          13580 non-null  int64         
 3   Type           13580 non-null  category      
 4   Price          13580 non-null  float64       
 5   Method         13580 non-null  category      
 6   SellerG        13580 non-null  category      
 7   Date           13580 non-null  datetime64[ns]
 8   Distance       13580 non-null  float64       
 9   Postcode       13580 non-null  int64         
 10  Bedroom        13580 non-null  float64       
 11  Bathroom       13580 non-null  float64       
 12  Car            13580 non-null  float64       
 13  Landsize       13580 non-null  float64       
 14  BuildingArea   13580 non-null  float64       
 15  CouncilArea    1221

None

1. Задали список столбцов, которые не берем в рассмотрение, а также условленный порог уникальных значений столбца;
2. В цикле перебираем имена столбцов, и, если число уникальных категорий меньше заданного максимального порога и имен столбцов нет в списке cols_to_exclude, то приводим столбец к типу данных category;
3. Итоговый объем памяти — 1.9+ Мб. Объем памяти таблицы уменьшился почти в 1.5 раза.

Особенно хорошо такое преобразование работает на действительно больших данных, где число строк превышает сотни тысяч или миллионы. Иногда изменение типов может уменьшить объем памяти в десятки раз и существенно увеличить производительность.

У типа данных category есть свой специальный аксессор cat, который позволяет получать информацию о своих значениях и преобразовывать их. С помощью атрибута этого аксессора categories мы можем получить список уникальных категорий столбца Regionname:

In [8]:
print(melb_df['Regionname'].cat.categories)

Index(['Eastern Metropolitan', 'Eastern Victoria', 'Northern Metropolitan',
       'Northern Victoria', 'South-Eastern Metropolitan',
       'Southern Metropolitan', 'Western Metropolitan', 'Western Victoria'],
      dtype='object')


In [9]:
# выводим, каким образом столбец кодируется в виде чисел памяти компьютера
display(melb_df['Regionname'].cat.codes)

0        2
1        2
2        2
3        2
4        2
        ..
13575    4
13576    6
13577    6
13578    6
13579    6
Length: 13580, dtype: int8

С помощью метода аксессора rename_categories() можно переименовать текущие значения категорий. Для этого в данный метод нужно передать словарь, ключи которого — старые имена категорий, а значения — новые.

In [10]:
melb_df['Type'] = melb_df['Type'].cat.rename_categories({
    'u': 'unit',
    't': 'townhouse',
    'h': 'house'
})
display(melb_df['Type'])

0        house
1        house
2        house
3        house
4        house
         ...  
13575    house
13576    house
13577    house
13578    house
13579    house
Name: Type, Length: 13580, dtype: category
Categories (3, object): ['house', 'townhouse', 'unit']

А теперь СИТУАЦИЯ: появилась новая партия домов и теперь мы продаем квартиры (flat). Создадим серию new_houses_types, в которой будем хранить типы зданий новой партии домов. Преобразуем тип new_houses_types в такой же тип, как и у столбца Type:

In [11]:
new_houses_types = pd.Series(['unit', 'house', 'flat', 'flat', 'house'])
new_houses_types = new_houses_types.astype(melb_df['Type'].dtype)
display(new_houses_types)

0     unit
1    house
2      NaN
3      NaN
4    house
dtype: category
Categories (3, object): ['house', 'townhouse', 'unit']

По какой-то причине вместо квартир мы получили пустые значения NaN.

Причина проста: тип данных category хранит только категории, которые были объявлены при его инициализации. При встрече с новой, неизвестной ранее категорией, этот тип превратит ее в пустое значение, так как он просто не знает о существовании этой категории.

Эта проблема решаема. Можно добавить категорию flat в столбец Type с помощью метода аксессора cat — add_categories(), в который достаточно положить имя новой категории:

In [12]:
melb_df['Type'] = melb_df['Type'].cat.add_categories('flat')
new_houses_types = pd.Series(['unit', 'house', 'flat', 'flat', 'house'])
new_houses_types = new_houses_types.astype(melb_df['Type'].dtype)
display(new_houses_types)

0     unit
1    house
2     flat
3     flat
4    house
dtype: category
Categories (4, object): ['house', 'townhouse', 'unit', 'flat']

Добавление новой категории не отразится на самом столбце — текущие категории не изменятся, однако такое преобразование позволит добавлять в таблицу новые данные о домах с новой категорией — flat.

Если набор категорий в столбце жестко не зафиксирован и может обновляться в процессе работы, то тип category не является подходящим типом данных для этого столбца или необходимо постоянно писать проверки при обновлении таблицы.

РЕКОМЕНДАЦИИ по использованию типа category:

1. Необязательно каждый раз преобразовывать категориальные данные в тип данных category. Зачастую это делается исключительно для оптимизации работы с большими данными.
2. Если набор данных занимает значительный процент используемой оперативной памяти, можно рассмотреть возможность использования типа category.
3. Если фиксируются очень серьезные проблемы с производительностью, то нужно обратить внимание на использование типа category.
4. Если решено использовать тип category, нужно быть осторожным при добавлении новой информации в таблицу. Нужно убедиться, что была собрана вся необходимая информация, произвести предобработку данных и только после этого использовать преобразование типов.

Задание:

1. Узнать, сколько памяти занимает таблица melb_df;
2. Преобразовать признак Suburb: оставить в столбце только 119 наиболее популярных пригородов, остальные заменить на other;
3. Привести данные в столбце Suburb к категориальному типу.

В качестве ответа записать разницу между объемом занимаемой памяти до преобразования и после него в Мб. Ответ округлить до десятых.

In [ ]:
old_size = melb_df.info()
popular_suburbs = melb_df['Suburb'].value_counts().nlargest(n=119).index
melb_df['Suburb'] = melb_df['Suburb'].apply(lambda x: x if x in popular_suburbs else 'other')
melb_df['Suburb'] = melb_df['Suburb'].astype('category')
new_size = melb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   index          13580 non-null  int64         
 1   Suburb         13580 non-null  category      
 2   Rooms          13580 non-null  int64         
 3   Type           13580 non-null  category      
 4   Price          13580 non-null  float64       
 5   Method         13580 non-null  category      
 6   SellerG        13580 non-null  category      
 7   Date           13580 non-null  datetime64[ns]
 8   Distance       13580 non-null  float64       
 9   Postcode       13580 non-null  int64         
 10  Bedroom        13580 non-null  float64       
 11  Bathroom       13580 non-null  float64       
 12  Car            13580 non-null  float64       
 13  Landsize       13580 non-null  float64       
 14  BuildingArea   13580 non-null  float64       
 15  CouncilArea    1221

TypeError: unsupported operand type(s) for -: 'NoneType' and 'NoneType'